In [ ]:
!pip install pymupdf pillow imagehash -q --break-system-packages

import fitz
import os
import io
import json
import imagehash
from PIL import Image
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PDF_PATH = "/content/drive/MyDrive/punto_eng.pdf"
OUTPUT_DIR = "/content/drive/MyDrive/extracted_icons_with_coords"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

doc = fitz.open(PDF_PATH)
print(f"Loaded: {len(doc)} pages")

In [ ]:
def extract_icons(doc, dpi=288,
                  first_pass_min=8,
                  second_pass_min=3, second_pass_max=7):

    mat = fitz.Matrix(dpi / 72, dpi / 72)
    icons = []

    for page_num, page in enumerate(doc):
        drawings = page.get_drawings()

        for i, drawing in enumerate(drawings):
            rect = drawing["rect"]
            if rect.is_empty or rect.is_infinite:
                continue

            n_paths = len(drawing["items"])
            w, h = rect.width, rect.height

            if w < 5 or h < 5:
                continue

            is_first = n_paths >= first_pass_min
            is_second = second_pass_min <= n_paths < first_pass_min

            if not (is_first or is_second):
                continue

            clip = rect + fitz.Rect(-2, -2, 2, 2)
            pix = page.get_pixmap(matrix=mat, clip=clip)
            img = Image.open(io.BytesIO(pix.tobytes("png")))

            icons.append({
                "img": img,
                "page": page_num + 1,
                "index": i,
                "n_paths": n_paths,
                "pass": "first" if is_first else "second",
                #og pdf coordinates
                "rect": [rect.x0, rect.y0, rect.x1, rect.y1]
            })

    first = sum(1 for x in icons if x["pass"] == "first")
    second = sum(1 for x in icons if x["pass"] == "second")
    print(f"Extracted {len(icons)} candidates ({first} first pass, {second} second pass)")
    return icons

raw_icons = extract_icons(doc)

In [ ]:
def normalize_icons(icons, target_size=128):

    for item in icons:
        img = item["img"].convert("RGBA")

        bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
        bg.paste(img, mask=img.split()[3])
        img = bg.convert("L")

        #binarize
        img = img.point(lambda x: 0 if x < 128 else 255, '1').convert("RGB")

        img.thumbnail((target_size, target_size), Image.LANCZOS)
        square = Image.new("RGB", (target_size, target_size), (255, 255, 255))
        offset = ((target_size - img.width) // 2, (target_size - img.height) // 2)
        square.paste(img, offset)

        item["img_normalized"] = square

    print(f"Normalized {len(icons)} icons")
    return icons

normalized_icons = normalize_icons(raw_icons)

In [ ]:
def deduplicate_icons(icons, hash_threshold=10):
    seen = {}
    unique = []

    for item in icons:
        h = imagehash.phash(item["img_normalized"])

        matched = None
        for seen_hash, uid in seen.items():
            if abs(h - seen_hash) <= hash_threshold:
                matched = uid
                break

        occurrence = {
            "page": item["page"],
            "rect": item["rect"],   #og pdf coordinates
            "n_paths": item["n_paths"]
        }

        if matched is None:

            seen[h] = len(unique)
            unique.append({
                "img_normalized": item["img_normalized"],
                "phash": str(h),
                "occurrences": [occurrence]
            })
        else:
            # add occurrence x duplicates
            unique[matched]["occurrences"].append(occurrence)

    print(f"\n   Deduplication done:")
    print(f"   Unique icons:     {len(unique)}")
    print(f"   Total occurrences: {sum(len(u['occurrences']) for u in unique)}")
    return unique

unique_icons = deduplicate_icons(normalized_icons)

In [ ]:
for i, item in enumerate(unique_icons):
    base_name = f"icon_{i+1:03d}"

    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    item["img_normalized"].save(png_path)

    json_path = os.path.join(OUTPUT_DIR, f"{base_name}.json")
    sidecar = {
        "name": base_name,
        "phash": item["phash"],
        "n_occurrences": len(item["occurrences"]),
        "occurrences": item["occurrences"]
    }
    with open(json_path, "w") as f:
        json.dump(sidecar, f, indent=2)

print(f"   Saved {len(unique_icons)} unique icons to {OUTPUT_DIR}")